# Research Notes Repository Chroma Query

In [22]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.research_library_document_loader import ResearchLibraryChromaDocumentLoader
from apps.agentic.core.constants import RESEARCH_LIBRARY_DB_NAME, RESEARCH_LIBRARY_COLLECTION_NAME
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = "../../.db"

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [23]:
RESEARCH_LIBRARY_DB_NAME, RESEARCH_LIBRARY_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('research_library',
 'research-library',
 '../../.db',
 ['.DS_Store', 'research_library', 'pdf_document_library', 'github'])

In [24]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [25]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)
print("total docs:", coll.count())

['research-library']
Opened: research-library
total docs: 417


In [26]:
title = "Analytic Mechanics"
res = coll.get(where={"title": title})   # no 'include' arg at all
print(f"{title} docs:", len(res["ids"]))

Analytic Mechanics docs: 87


## Verify Document Metadata

In [27]:
probe = coll.get(limit=5000)  # adjust as needed
metas  = [m for m in probe.get("metadatas", []) if m]
len(metas)

417

In [28]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
print("metadata keys:", sorted(keys))

metadata keys: ['authors', 'date', 'ext', 'filename', 'h1', 'h2', 'images', 'path', 'section', 'section_char_offset', 'start_index', 'tags', 'title', 'topic']


In [29]:
print("title counts:", Counter(m.get("title") for m in metas if "title" in m and m.get("title")).most_common(10))
print("authors counts:", Counter(m.get("authors") for m in metas if "authors" in m and m.get("authors")).most_common(10))

title counts: [('Autoregressive Processes', 278), ('Analytic Mechanics', 87), ('The Evolution of Carnot’s Principle', 52)]
authors counts: [('Troy Stribling', 365), ('E. T. Jaynes', 52)]


## Search Validation

### Similarity

In [12]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore

docs = vs.similarity_search("Derivation of Lagrange Multipliers", k=5, filter={"title": "Analytic Mechanics"})
for d in docs:
    print(d.page_content[:160].replace("\n"," "), "\n")

The method of Lagrange multipliers used to determithe the constraing forces acting on a system.   Consider a system described by $n$ generalized coordinates $q_ 

The total time derivative of the Lagrangian,   $$ L\left(q_{1}, q_{2}, \ldots, q_{n} ; \dot{q}_{1}, \dot{q}_{2}, \ldots, q_{n}, t\right) $$   is given by   $$ \ 

$$ \frac{d}{d t}\left(\frac{\partial L}{\partial \dot{q}_{j}}\right)-\frac{\partial L}{\partial \varphi_{j}}=0 $$   $L$ is called the Lagrangian and the equatio 

Now   $$ \begin{aligned} H & =\sum_{i=1}^{n} \dot{q}_{i} p_{i}-L \\ \Rightarrow L & =\sum_{i=1}^{n} \dot{q}_{i} p_{i}-H \end{aligned} $$   Hamitor's Pricipal be 

$$   Here the sum is over $N$ particles not coordinates. lagranges equation for a particle   $$ \frac{d}{d t}\left(\frac{\partial L}{\partial \bar{v}_{i}}\right 



### MMR Search

In [36]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 50, "filter": {"title": "Analytic Mechanics"}}
)

hits = retriever.invoke("Derivation of Lagrange Multipliers")
for d in hits[:10]:
    print(d.metadata.get("path"), d.metadata.get("section"), d.metadata.get("section_char_offset"))

/Users/troy/Develop/gly.fish/yada/research_library/Troy Notes/analytic-mechanics-2-e26f51a6-20e3-4f03-a441-234ae51f5317.md 9 225
/Users/troy/Develop/gly.fish/yada/research_library/Troy Notes/analytic-mechanics-2-e26f51a6-20e3-4f03-a441-234ae51f5317.md 9 1225
/Users/troy/Develop/gly.fish/yada/research_library/Troy Notes/analytic-mechanics-2-e26f51a6-20e3-4f03-a441-234ae51f5317.md 9 625
/Users/troy/Develop/gly.fish/yada/research_library/Troy Notes/analytic-mechanics-2-e26f51a6-20e3-4f03-a441-234ae51f5317.md 4 537
/Users/troy/Develop/gly.fish/yada/research_library/Troy Notes/analytic-mechanics-2-e26f51a6-20e3-4f03-a441-234ae51f5317.md 3 11
/Users/troy/Develop/gly.fish/yada/research_library/Troy Notes/analytic-mechanics-2-e26f51a6-20e3-4f03-a441-234ae51f5317.md 29 1989
/Users/troy/Develop/gly.fish/yada/research_library/Troy Notes/analytic-mechanics-2-e26f51a6-20e3-4f03-a441-234ae51f5317.md 22 829
/Users/troy/Develop/gly.fish/yada/research_library/Troy Notes/analytic-mechanics-2-e26f51a6-20

## Search Examples 

In [34]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"title": "Analytic Mechanics"},
    },
)
hits = retriever.invoke("Derivation of Lagrange Multipliers")

[print(h) for h in hits]

page_content='The method of Lagrange multipliers used to determithe the constraing forces acting on a system.  
Consider a system described by $n$ generalized coordinates $q_{1}, q_{2}, \ldots, q_{n}$. Furthure assume that the system has $k$ constraints acting on it. The constraints eleminate $k$ of the generalized coordinates leaving a total $n-k$ linearly independent coordinates.
The time behavior of the system is described by Hamilton's Principal  
$$
\delta I=\delta \int_{t_{s}}^{t_{f}} L d t=\int_{t_{s}}^{t_{f}} \delta L d t=0
$$  
with $k$ equations defining holonomic constraints,  
$$
f_{j}\left(q_{1}, q_{2}, \ldots, q_{n}\right)=0 \quad \forall \quad j=1,2, \ldots, k
$$  
consider the variation of the constraints  
$$
\delta f_{j}=\sum_{l=1}^{n} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$$  
For each $f_{j}$ define a scalar parameter $\lambda_{j}$ such that  
$$
\lambda_{j} \delta f_{j}=\sum_{l=1}^{n} \lambda_{j} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$

[None, None, None, None, None, None, None, None]

In [38]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"title": "The Evolution of Carnot’s Principle"},
    },
)
hits = retriever.invoke("Search for Lord Kelvin's contribution to the second law of thermodynamics.")

[print(h) for h in hits]

InternalError: Error executing plan: Internal error: Error finding id

### Use MMR Call Directly

In [18]:
docs = vs.max_marginal_relevance_search(
    "Derivation of Lagrange Multipliers",
    k=8,
    fetch_k=60,
    filter={"$and": [{"title": "Analytic Mechanics"}, {"authors": "Troy Stribling"}]},
)

[print(h) for h in docs]

page_content='The method of Lagrange multipliers used to determithe the constraing forces acting on a system.  
Consider a system described by $n$ generalized coordinates $q_{1}, q_{2}, \ldots, q_{n}$. Furthure assume that the system has $k$ constraints acting on it. The constraints eleminate $k$ of the generalized coordinates leaving a total $n-k$ linearly independent coordinates.
The time behavior of the system is described by Hamilton's Principal  
$$
\delta I=\delta \int_{t_{s}}^{t_{f}} L d t=\int_{t_{s}}^{t_{f}} \delta L d t=0
$$  
with $k$ equations defining holonomic constraints,  
$$
f_{j}\left(q_{1}, q_{2}, \ldots, q_{n}\right)=0 \quad \forall \quad j=1,2, \ldots, k
$$  
consider the variation of the constraints  
$$
\delta f_{j}=\sum_{l=1}^{n} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$$  
For each $f_{j}$ define a scalar parameter $\lambda_{j}$ such that  
$$
\lambda_{j} \delta f_{j}=\sum_{l=1}^{n} \lambda_{j} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$

[None, None, None, None, None, None, None, None]